# Prepare Calendar Table

Generates a standard date dimension table `adb_dev_healthcare.silver.calendar` spanning **1940-01-01** to **2030-12-31**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Generate date sequence from 1940-01-01 to 2030-12-31
df = (
    spark.range(0, 33238)  # 33238 days from 1940-01-01 to 2030-12-31
    .select(F.expr("date_add('1940-01-01', CAST(id AS INT))").alias("full_date"))
    .filter(F.col("full_date") <= "2030-12-31")
)

calendar_df = df.select(
    # Key & Date
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("full_date"),

    # Year
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.date_format("full_date", "MMM").alias("month_short_name"),
    F.weekofyear("full_date").alias("week_of_year"),

    # Day
    F.dayofmonth("full_date").alias("day_of_month"),
    F.dayofweek("full_date").alias("day_of_week"),  # 1=Sun, 7=Sat
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.date_format("full_date", "EEE").alias("day_short_name"),
    F.dayofyear("full_date").alias("day_of_year"),

    # Flags
    F.when(F.dayofweek("full_date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("full_date").isin(1, 7), False).otherwise(True).alias("is_weekday"),
    F.when(
        (F.year("full_date") % 4 == 0) &
        ((F.year("full_date") % 100 != 0) | (F.year("full_date") % 400 == 0)),
        True
    ).otherwise(False).alias("is_leap_year"),

    # Formatted strings
    F.date_format("full_date", "yyyy-MM").alias("year_month"),
    F.concat(F.year("full_date"), F.lit("-Q"), F.quarter("full_date")).alias("year_quarter"),

    # Month boundaries
    F.trunc("full_date", "MM").alias("first_day_of_month"),
    F.last_day("full_date").alias("last_day_of_month"),

    # Quarter boundaries
    F.trunc("full_date", "quarter").alias("first_day_of_quarter"),
    F.date_sub(
        F.add_months(F.trunc("full_date", "quarter"), 3), 1
    ).alias("last_day_of_quarter"),

    # Year boundaries
    F.trunc("full_date", "year").alias("first_day_of_year"),
    F.to_date(F.concat(F.year("full_date"), F.lit("-12-31"))).alias("last_day_of_year"),

    # Week of month
    F.ceil(F.dayofmonth("full_date") / 7).cast("int").alias("week_of_month"),

    # Fiscal year (assuming fiscal year starts in April)
    F.when(F.month("full_date") >= 4, F.year("full_date") + 1)
     .otherwise(F.year("full_date")).alias("fiscal_year"),
    F.when(F.month("full_date") >= 4, F.quarter(F.add_months("full_date", -3)))
     .otherwise(F.quarter(F.add_months("full_date", -3)) + 4)
     .alias("fiscal_quarter"),

    # Half year
    F.when(F.month("full_date") <= 6, 1).otherwise(2).alias("half_year"),

    # Days in month
    F.dayofmonth(F.last_day("full_date")).alias("days_in_month"),
)

# Write to Silver layer
calendar_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("adb_dev_healthcare.silver.calendar")

print(f"Calendar table created with {calendar_df.count()} rows.")
display(spark.table("adb_dev_healthcare.silver.calendar").limit(10))